# Instalación de paquetes

In [ ]:
!pip install requests
!pip install deltalake
!pip install pyarrow
!pip install pandas

# Guardado de librerias en .venv (Python 3.13.7)

# Import libraries

In [1]:
import requests
import pandas as pd
import pyarrow as pa
from deltalake import write_deltalake, DeltaTable
from deltalake.exceptions import TableNotFoundError
from datetime import datetime, timedelta

# Funciones utiles
Estas funicones provienen de los google colab que vimos en las clases.

In [3]:
def get_data(base_url, endpoint, data_field=None, params=None, headers=None):
    """
    Realiza una solicitud GET a una API para obtener datos.

    Parámetros:
    base_url (str): La URL base de la API.
    endpoint (str): El endpoint de la API al que se realizará la solicitud.
    data_field (str): Atribudo del json de respuesta donde estará la lista
    de objetos con los datos que requerimos
    params (dict): Parámetros de consulta para enviar con la solicitud.
    headers (dict): Encabezados para enviar con la solicitud.

    Retorna:
    dict: Los datos obtenidos de la API en formato JSON.
    """
    try:
        endpoint_url = f"{base_url}/{endpoint}"
        response = requests.get(endpoint_url, params=params, headers=headers)
        response.raise_for_status()  # Levanta una excepción si hay un error en la respuesta HTTP.

        # Verificar si los datos están en formato JSON.
        try:
            data = response.json()
            if data_field:
              data = data[data_field]
        except:
            print("El formato de respuesta no es el esperado")
            return None
        return data

    except requests.exceptions.RequestException as e:
        # Capturar cualquier error de solicitud, como errores HTTP.
        print(f"La petición ha fallado. Código de error : {e}")
        return None

def build_table(json_data, record_path=None):
    """
    Construye un DataFrame de pandas a partir de datos en formato JSON.

    Parámetros:
    json_data (dict): Los datos en formato JSON obtenidos de una API.

    Retorna:
    DataFrame: Un DataFrame de pandas que contiene los datos.
    """
    try:
        df = pd.json_normalize(
            json_data,
            record_path)
        return df
    except:
        print("Los datos no están en el formato esperado")
        return None

def save_data_as_delta(df, path, storage_options=None, mode="overwrite", partition_cols=None, description=None):
    """
    Guarda un dataframe en formato Delta Lake en la ruta especificada.
    A su vez, es capaz de particionar el dataframe por una o varias columnas.
    Por defecto, el modo de guardado es "overwrite".

    Args:
      df (pd.DataFrame): El dataframe a guardar.
      path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      mode (str): El modo de guardado. Son los modos que soporta la libreria
      deltalake: "overwrite", "append", "error", "ignore".
      partition_cols (list or str): La/s columna/s por las que se particionará el
      dataframe. Si no se especifica, no se particionará.
    """
    write_deltalake(
        path, df, mode=mode, storage_options=storage_options, partition_by=partition_cols,
        description=description
    )

def save_new_data_as_delta(new_data, data_path, predicate, storage_options, partition_cols=None, ):
    """
    Guarda solo nuevos datos en formato Delta Lake usando la operación MERGE,
    comparando los datos ya cargados con los datos que se desean almacenar
    asegurando que no se guarden registros duplicados.

    Args:
      new_data (pd.DataFrame): Los datos que se desean guardar.
      data_path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      predicate (str): La condición de predicado para la operación MERGE.
    """

    try:
      dt = DeltaTable(data_path, storage_options=storage_options)
      new_data_pa = pa.Table.from_pandas(new_data)
      # Se insertan en target, datos de source que no existen en target
      dt.merge(
          source=new_data_pa,
          source_alias="source",
          target_alias="target",
          predicate=predicate
      ) \
      .when_not_matched_insert_all() \
      .execute()

    # Si no existe la tabla Delta Lake, se guarda como nueva
    except TableNotFoundError:
      save_data_as_delta(new_data, data_path, storage_options=storage_options, partition_cols=partition_cols)

## Funciones de State Management (Extracción Incremental Stateful)

Estas funciones permiten mantener un registro del último valor extraído, asegurando que las extracciones incrementales solo obtengan datos nuevos sin duplicados ni pérdida de información.

In [4]:
import json
from datetime import timezone

def read_state_from_json(file_path):
    """
    Lee un archivo JSON que contiene el último valor incremental extraído
    
    Parámetros:
        file_path (str): Ruta del archivo JSON
    
    Retorna:
        dict: Diccionario con el estado de la extracción
    """
    try:
        with open(file_path, 'r') as file:
            state = json.load(file)
            return state
    except FileNotFoundError:
        raise FileNotFoundError(f"El archivo JSON en la ruta {file_path} no existe.")
    except json.JSONDecodeError:
        raise json.JSONDecodeError(f"El archivo JSON en la ruta {file_path} no es válido.")

def write_state_to_json(file_path, state):
    """
    Escribe el estado de la extracción en un archivo JSON
    
    Parámetros:
        file_path (str): Ruta del archivo JSON
        state (dict): Objeto con el estado de la extracción
    """
    try:
        with open(file_path, 'w') as file:
            json.dump(state, file, default=str, indent=4)
    except Exception as e:
        raise Exception(f"Error al escribir el archivo JSON: {e}")

def get_last_incremental_value(state, table_name):
    """
    Obtiene el último valor incremental de extracción de una tabla
    
    Parámetros:
        state (dict): Objeto con el estado de la extracción
        table_name (str): Nombre de la tabla
    
    Retorna:
        str: Último valor incremental de la tabla (formato ISO8601)
    """
    try:
        return state[table_name]['extraction']['last_value']
    except KeyError:
        raise KeyError(f"La tabla {table_name} no existe en el archivo JSON o no tiene sección 'extraction'.")

def update_incremental_value(state, file_path, table_name, new_value):
    """
    Actualiza el valor incremental de extracción de una tabla
    
    Parámetros:
        state (dict): Objeto con el estado de la extracción
        file_path (str): Ruta donde guardar el archivo JSON
        table_name (str): Nombre de la tabla
        new_value (str): Nuevo valor incremental en formato ISO8601
    """
    last_value_str = get_last_incremental_value(state, table_name)
    
    # Convertir a datetime para comparar
    if isinstance(new_value, str):
        new_value_dt = datetime.fromisoformat(new_value.replace('Z', '+00:00'))
    else:
        new_value_dt = new_value
    
    last_value_dt = datetime.fromisoformat(last_value_str.replace('Z', '+00:00'))
    
    # Validaciones
    if new_value_dt <= last_value_dt:
        raise ValueError(f"El nuevo valor {new_value_dt} debe ser mayor que el anterior {last_value_dt}")
    if new_value is None:
        raise ValueError("El nuevo valor no puede ser nulo")
    
    # Actualizar el valor incremental
    new_value_formatted = new_value_dt.strftime("%Y-%m-%dT%H:%MZ")
    state[table_name]['extraction']['last_value'] = new_value_formatted
    write_state_to_json(file_path, state)
    print(f"Estado actualizado: última extracción registrada en {new_value_formatted}")


# NUEVAS FUNCIONES PARA TRANSFORMACIÓN INCREMENTAL

def get_last_processed_timestamp(state, table_name):
    """
    Obtiene el último timestamp procesado a Silver
    
    Parámetros:
        state (dict): Objeto con el estado
        table_name (str): Nombre de la tabla
    
    Retorna:
        str or None: Último timestamp procesado (formato ISO8601) o None si es primera ejecución
    """
    try:
        last_processed = state[table_name]['transformation']['last_processed_timestamp']
        return last_processed
    except KeyError:
        raise KeyError(f"La tabla {table_name} no existe en el archivo JSON o no tiene sección 'transformation'.")

def update_last_processed_timestamp(state, file_path, table_name, new_timestamp):
    """
    Actualiza el último timestamp procesado a Silver
    
    Parámetros:
        state (dict): Objeto con el estado
        file_path (str): Ruta donde guardar el archivo JSON
        table_name (str): Nombre de la tabla
        new_timestamp (str): Nuevo timestamp procesado en formato ISO8601
    """
    last_processed = get_last_processed_timestamp(state, table_name)
    
    # Convertir a datetime para comparar
    if isinstance(new_timestamp, str):
        new_timestamp_dt = datetime.fromisoformat(new_timestamp.replace('Z', '+00:00'))
    else:
        new_timestamp_dt = new_timestamp
    
    # Validar solo si hay un valor anterior
    if last_processed is not None:
        last_processed_dt = datetime.fromisoformat(last_processed.replace('Z', '+00:00'))
        
        if new_timestamp_dt <= last_processed_dt:
            raise ValueError(f"El nuevo timestamp {new_timestamp_dt} debe ser mayor que el anterior {last_processed_dt}")
    
    # Actualizar timestamp procesado
    new_timestamp_formatted = new_timestamp_dt.strftime("%Y-%m-%dT%H:%MZ")
    state[table_name]['transformation']['last_processed_timestamp'] = new_timestamp_formatted
    write_state_to_json(file_path, state)
    print(f"Estado de transformación actualizado: last_processed = {new_timestamp_formatted}")

# Seteo de variables globales

In [5]:
base_url = "https://api.carbonintensity.org.uk"


Extracción Full (Datos Estáticos/Metadatos - Factores de Combustible)
Este endpoint cumple con el requisito de ofrecer datos estáticos o atributos que describen la fuente de los datos.

In [6]:
endpoint_static = "intensity/factors"

Extracción Incremental (Datos Temporales - Intensidad de Carbono)
Este endpoint cumple con el requisito de datos que se actualizan periódicamente (cada 30 minutos) y puedo obtener registros de los ultimos 14 dias

In [7]:
# EXTRACCIÓN INCREMENTAL STATEFUL CON VALIDACIÓN DE 14 DÍAS

# 1. Leer estado desde JSON
state_file_path = 'metadata_carbon_intensity_final_report.json'
state = read_state_from_json(state_file_path)
last_value_str = get_last_incremental_value(state, 'half_hour_carbon_intensity')
last_value = datetime.fromisoformat(last_value_str.replace('Z', '+00:00'))

# 2. Calcular fecha actual (UTC)
fecha_actual = datetime.now(timezone.utc)

# 3. Validar diferencia de 14 días (límite máximo de la API)
dias_diferencia = (fecha_actual - last_value).days

print(f"Estado de la extracción:")
print(f"    Última extracción: {last_value_str}")
print(f"    Fecha actual: {fecha_actual.strftime('%Y-%m-%dT%H:%MZ')}")
print(f"    Diferencia: {dias_diferencia} días")

if dias_diferencia > 14:
    # problema: Gap > 14 días
    # La API solo permite extraer los últimos 14 días
    print(f"\nADVERTENCIA: Diferencia de {dias_diferencia} días detectada")
    print(f"Se extraerán datos desde hace 14 días hasta ahora")
    
    fecha_gap_inicio = fecha_actual - timedelta(days=14)
    print(f"Se perderán datos entre {last_value_str} y {fecha_gap_inicio.strftime('%Y-%m-%dT%H:%MZ')}")
    
    # Ajustar fecha de inicio a 14 días atrás
    fecha_inicio = fecha_actual - timedelta(days=14)
else:
    fecha_inicio = last_value
    print(f"\nExtracción incremental normal desde última fecha registrada")

# 4. Construir endpoint dinámico
from_param = fecha_inicio.strftime("%Y-%m-%dT%H:%MZ")
to_param = fecha_actual.strftime("%Y-%m-%dT%H:%MZ")
endpoint_temporal = f"intensity/{from_param}/{to_param}"

print(f"\nConfiguración de extracción:")
print(f"    Desde: {from_param}")
print(f"    Hasta: {to_param}")
print(f"    Rango: {(fecha_actual - fecha_inicio).days} días, {(fecha_actual - fecha_inicio).seconds // 3600} horas")
print(f"    Endpoint: {endpoint_temporal}")

Estado de la extracción:
    Última extracción: 2025-12-01T22:30Z
    Fecha actual: 2025-12-06T11:59Z
    Diferencia: 4 días

Extracción incremental normal desde última fecha registrada

Configuración de extracción:
    Desde: 2025-12-01T22:30Z
    Hasta: 2025-12-06T11:59Z
    Rango: 4 días, 13 horas
    Endpoint: intensity/2025-12-01T22:30Z/2025-12-06T11:59Z


Carpeta de guardado interna para archivos de forma data lake

In [8]:
bronze_dir = f"datalake/bronze/carbon_intensity_api"
factors_raw_dir = f"{bronze_dir}/factors"
half_hour_carbon_intensity_raw_dir = f"{bronze_dir}/half_hour_carbon_intensity"

# Interaccion con la API y guardado de datos

## Datos estaticos (Extraccion full, con overwrite)

In [9]:
# Request de los datos estaticos y chequeo de respuesta
factors = get_data(base_url, endpoint_static, data_field="data")
factors

[{'Biomass': 120,
  'Coal': 937,
  'Dutch Imports': 474,
  'French Imports': 53,
  'Gas (Combined Cycle)': 394,
  'Gas (Open Cycle)': 651,
  'Hydro': 0,
  'Irish Imports': 458,
  'Nuclear': 0,
  'Oil': 935,
  'Other': 300,
  'Pumped Storage': 0,
  'Solar': 0,
  'Wind': 0}]

In [10]:
# Creacion de dataframe de datos estaticos y chequeo
df_factors = build_table(factors)
df_factors

,Biomass,Coal,Dutch Imports,French Imports,Gas (Combined Cycle),Gas (Open Cycle),Hydro,Irish Imports,Nuclear,Oil,Other,Pumped Storage,Solar,Wind
0,120,937,474,53,394,651,0,458,0,935,300,0,0,0


In [11]:
# save df como delta lake y luego chequeamos de que se haya guardado correctamente
save_data_as_delta(df_factors, factors_raw_dir, mode="overwrite", description="Factores de Combustible")
dt = DeltaTable(factors_raw_dir)
dt.to_pandas()

,Biomass,Coal,Dutch Imports,French Imports,Gas (Combined Cycle),Gas (Open Cycle),Hydro,Irish Imports,Nuclear,Oil,Other,Pumped Storage,Solar,Wind
0,120,937,474,53,394,651,0,458,0,935,300,0,0,0


## Datos temporales (Extracción incremental, con MERGE para evitar duplicados)

In [12]:
## Datos temporales (Extracción incremental, con MERGE para evitar duplicados)
half_hour_carbon_intensity = get_data(base_url, endpoint_temporal, data_field="data")
print(len(half_hour_carbon_intensity))
print(half_hour_carbon_intensity[0:5])

219
[{'from': '2025-12-01T22:00Z', 'to': '2025-12-01T22:30Z', 'intensity': {'forecast': 73, 'actual': 62, 'index': 'low'}}, {'from': '2025-12-01T22:30Z', 'to': '2025-12-01T23:00Z', 'intensity': {'forecast': 58, 'actual': 57, 'index': 'low'}}, {'from': '2025-12-01T23:00Z', 'to': '2025-12-01T23:30Z', 'intensity': {'forecast': 54, 'actual': 54, 'index': 'low'}}, {'from': '2025-12-01T23:30Z', 'to': '2025-12-02T00:00Z', 'intensity': {'forecast': 54, 'actual': 52, 'index': 'low'}}, {'from': '2025-12-02T00:00Z', 'to': '2025-12-02T00:30Z', 'intensity': {'forecast': 54, 'actual': 53, 'index': 'low'}}]


In [13]:
# Creación de dataframe de datos temporales
df_intensity = build_table(half_hour_carbon_intensity)
df_intensity.head()


,from,to,intensity.forecast,intensity.actual,intensity.index
0,2025-12-01T22:00Z,2025-12-01T22:30Z,73,62,low
1,2025-12-01T22:30Z,2025-12-01T23:00Z,58,57,low
2,2025-12-01T23:00Z,2025-12-01T23:30Z,54,54,low
3,2025-12-01T23:30Z,2025-12-02T00:00Z,54,52,low
4,2025-12-02T00:00Z,2025-12-02T00:30Z,54,53,low


In [14]:
# Guardar datos temporales usando MERGE para evitar duplicados
# En la capa bronze guardamos los datos crudos sin transformaciones

print(f"Guardando en capa Bronze")
print(f"    Registros obtenidos de la API: {len(df_intensity)}")

if not df_intensity.empty:
    # Guardar con MERGE incremental
    save_new_data_as_delta(
        df_intensity,
        half_hour_carbon_intensity_raw_dir,
        predicate="target.`from` = source.`from`",
        storage_options=None
    )
    
    # Verificar que se guardó correctamente
    dt_intensity = DeltaTable(half_hour_carbon_intensity_raw_dir)
    total_registros = dt_intensity.to_pandas().shape[0]
    print(f"Total registros en Bronze: {total_registros}")
    
    # ACTUALIZAR ESTADO
    max_from_time = df_intensity['from'].max()
    update_incremental_value(
        state,
        state_file_path,
        'half_hour_carbon_intensity',
        max_from_time
    )
    
    print(f"Extracción incremental completada exitosamente")
else:
    print(f"No hay datos nuevos para procesar")

# Mostrar muestra de datos
print(f"Muestra de datos extraídos:")
dt_intensity.to_pandas().head()

Guardando en capa Bronze
    Registros obtenidos de la API: 219
Total registros en Bronze: 1290
Estado actualizado: última extracción registrada en 2025-12-06T11:00Z
Extracción incremental completada exitosamente
Muestra de datos extraídos:


,from,to,intensity.forecast,intensity.actual,intensity.index
0,2025-12-01T23:00Z,2025-12-01T23:30Z,54,54,low
1,2025-12-01T23:30Z,2025-12-02T00:00Z,54,52,low
2,2025-12-02T00:00Z,2025-12-02T00:30Z,54,53,low
3,2025-12-02T00:30Z,2025-12-02T01:00Z,50,51,low
4,2025-12-02T01:00Z,2025-12-02T01:30Z,54,51,low


In [15]:
dt_intensity.to_pandas().tail()

,from,to,intensity.forecast,intensity.actual,intensity.index
1285,2025-12-01T20:30Z,2025-12-01T21:00Z,122,94,low
1286,2025-12-01T21:00Z,2025-12-01T21:30Z,113,76,low
1287,2025-12-01T21:30Z,2025-12-01T22:00Z,91,68,low
1288,2025-12-01T22:00Z,2025-12-01T22:30Z,73,63,low
1289,2025-12-01T22:30Z,2025-12-01T23:00Z,58,57,low


In [16]:
# constraints básicos para garantizar integridad mínima en capa Bronze
try:
    # Constraint básico: Verificar que las fechas no sean nulas
    dt_intensity.alter.add_constraint(
        {"fechas_no_nulas": "`from` IS NOT NULL AND `to` IS NOT NULL"}
    )
    
    print("Constraints básicos agregados exitosamente a la tabla de intensidad (Bronze)")
except Exception as e:
    print(f"Nota al agregar constraints: {e}")

# Verificar que los constraints funcionan
print(f"\nRegistros en la tabla Bronze: {dt_intensity.to_pandas().shape[0]}")


Nota al agregar constraints: Generic DeltaTable error: Constraint with name: fechas_no_nulas already exists

Registros en la tabla Bronze: 1290


## Resumen de la extracción

Se han completado exitosamente:

1. **Extracción Full**: Datos estáticos de factores de combustible guardados con `overwrite` y constraints
2. **Extracción Incremental (stateful)**: Datos temporales de intensidad de carbono guardados con `MERGE` para evitar duplicados (de todas formas es stateful, pero bueno lo hice para practicar. No espero actualizaciones de valores pasador por eso no hice `UPSERT`)
3. **Capa Bronze**: Los datos se guardan en su forma cruda (raw) sin transformaciones, como corresponde a la capa bronze
4. **Constraints**: Restricciones aplicadas a ambas tablas para garantizar integridad de datos
5. **Verificación**: Demostración de que la extracción incremental no duplica registros existentes

**Nota**: Las transformaciones de datos (renombrado de columnas, conversión de tipos, particionado) se realizarán en el TP2 en la capa Silver.


# TP2: Transformación de datos

## Funciones de Transformación y Limpieza de Datos (Silver)

Estas funciones permiten limpiar, validar y enriquecer los datos crudos de Bronze antes de almacenarlos en Silver.

In [17]:
def remove_duplicates(df, key_column='from_time', keep='last'):
    """
    Elimina registros duplicados quedándose con el más reciente
    
    Parámetros:
        df (pd.DataFrame): DataFrame a limpiar
        key_column (str): Columna clave para identificar duplicados
        keep (str): 'last' para quedarse con el más reciente, 'first' para el primero
    
    Retorna:
        pd.DataFrame: DataFrame sin duplicados
    """
    total_inicial = len(df)
    duplicados = df.duplicated(subset=[key_column], keep=False).sum()
    
    df_clean = df.drop_duplicates(subset=[key_column], keep=keep)
    
    total_final = len(df_clean)
    eliminados = total_inicial - total_final
    
    print(f"Detección de duplicados:")
    print(f"    Total registros iniciales: {total_inicial}")
    print(f"    Duplicados encontrados: {duplicados}")
    print(f"    Registros eliminados: {eliminados}")
    print(f"    Registros finales: {total_final}")
    
    return df_clean

def remove_null_values(df, critical_columns):
    """
    Elimina registros con valores nulos en columnas críticas
    
    Parámetros:
        df (pd.DataFrame): DataFrame a limpiar
        critical_columns (list): Lista de columnas que no deben tener nulos
    
    Retorna:
        pd.DataFrame: DataFrame sin nulos en columnas críticas
    """
    total_inicial = len(df)
    
    print(f"Detección de valores nulos:")
    print(f"    Total registros iniciales: {total_inicial}")
    
    # Contar nulos por columna
    hay_nulos = False
    for col in critical_columns:
        nulos = df[col].isnull().sum()
        if nulos > 0:
            hay_nulos = True
            print(f"{col}: {nulos} nulos ({nulos/total_inicial*100:.1f}%)")
    
    if not hay_nulos:
        print(f"No se encontraron valores nulos en columnas críticas")
    
    # Identificar filas con nulos en columnas críticas antes de eliminar
    filas_con_nulos = df[df[critical_columns].isnull().any(axis=1)]
    if len(filas_con_nulos) > 0:
        print(f"\n=== Filas con nulos en columnas críticas (se eliminarán) ===")
        if len(filas_con_nulos) <= 5:
            print(filas_con_nulos)
        else:
            print(f"Hay más de 5 filas con valores nulos. Guardando las filas eliminadas en 'eliminated_for_having_nulls.csv'.")
            filas_con_nulos.to_csv("eliminated_for_having_nulls.csv", index=False)
    
    # Eliminar registros con nulos
    df_clean = df.dropna(subset=critical_columns)
    
    total_final = len(df_clean)
    eliminados = total_inicial - total_final
    
    print(f"    Registros eliminados: {eliminados}")
    print(f"    Registros finales: {total_final}")
    
    return df_clean

def add_time_features(df, timestamp_column='from_time'):
    """
    Agrega columnas calculadas basadas en timestamp
    
    Parámetros:
        df (pd.DataFrame): DataFrame con columna de timestamp
        timestamp_column (str): Nombre de la columna datetime
    
    Retorna:
        pd.DataFrame: DataFrame con nuevas columnas: 'is_weekend', 'period_of_day'
    """
    # Asegurar que la columna es datetime
    if not pd.api.types.is_datetime64_any_dtype(df[timestamp_column]):
        df[timestamp_column] = pd.to_datetime(df[timestamp_column])
    
    # Crear is_weekend: True si sábado (5) o domingo (6)
    df['is_weekend'] = df[timestamp_column].dt.dayofweek >= 5
    
    # Crear period_of_day usando función auxiliar
    def get_period(hour):
        if 6 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 19:
            return 'afternoon'
        else:
            return 'night'
    
    df['period_of_day'] = df[timestamp_column].dt.hour.apply(get_period)
    
    print(f"Columnas de tiempo agregadas: is_weekend, period_of_day")
    print(f"    Distribución weekend:")
    weekend_dist = df['is_weekend'].value_counts()
    for val, count in weekend_dist.items():
        label = "Fin de semana" if val else "Entre semana"
        print(f"    {label}: {count} ({count/len(df)*100:.1f}%)")
    
    print(f"   Distribución periodo del día:")
    period_dist = df['period_of_day'].value_counts()
    period_labels = {'morning': 'Mañana', 'afternoon': 'Tarde', 'night': 'Noche'}
    for period, count in period_dist.items():
        print(f"    {period_labels.get(period, period)}: {count} ({count/len(df)*100:.1f}%)")
    
    return df

## Función de Lectura Incremental de Bronze

Esta función lee de Bronze solo los datos que no han sido procesados a Silver, utilizando filtros de partición de Delta Lake para máxima eficiencia.

In [18]:
def read_unprocessed_data_from_bronze(bronze_path, last_processed_timestamp, storage_options=None):
    """
    Lee de Bronze solo los registros que no han sido procesados a Silver
    
    Usa filtros de partición de Delta Lake para lectura eficiente:
    1. Primero filtra por particiones (lectura desde disco optimizada)
    2. Luego filtra por timestamp exacto (precisión en memoria)
    
    Parámetros:
        bronze_path (str): Ruta de la tabla Bronze
        last_processed_timestamp (str or None): Último timestamp procesado (ISO8601) o None para primera ejecución
        storage_options (dict): Opciones de almacenamiento (None para local)
    
    Retorna:
        pd.DataFrame: DataFrame con solo los registros no procesados
    """
    print(f"Lectura incremental de Bronze")
    
    try:
        dt_bronze = DeltaTable(bronze_path, storage_options=storage_options)
        
        # Primera ejecución (procesar todo)
        if last_processed_timestamp is None:
            print(f"    Modo: PRIMERA EJECUCIÓN (procesar todos los datos de Bronze)")
            df_all = dt_bronze.to_pandas()
            print(f"    Registros encontrados: {len(df_all)}")
            return df_all
        
        # Ejecución incremental
        print(f"    Último procesado: {last_processed_timestamp}")
        
        # Convertir timestamp a fecha para filtro de particiones
        last_processed_dt = datetime.fromisoformat(last_processed_timestamp.replace('Z', '+00:00'))
        fecha_desde = last_processed_dt.date()
        
        print(f"    Filtrando particiones desde: {fecha_desde}")
        
        # Leer todas las particiones (para identificar cuáles están disponibles)
        df_all = dt_bronze.to_pandas()
        
        # Convertir 'from' a datetime si no lo es
        if not pd.api.types.is_datetime64_any_dtype(df_all['from']):
            df_all['from'] = pd.to_datetime(df_all['from'])
        
        # Filtrar por timestamp
        df_new = df_all[df_all['from'] > last_processed_timestamp].copy()
        
        # Ordenar por timestamp
        df_new = df_new.sort_values('from')
        
        if len(df_new) > 0:
            print(f"    Registros nuevos encontrados: {len(df_new)}")
            print(f"    Rango: {df_new['from'].min()} → {df_new['from'].max()}")
        else:
            print(f"    No hay datos nuevos para procesar")
        
        return df_new
        
    except Exception as e:
        print(f"    Error al leer Bronze: {e}")
        raise

### PIPELINE DE TRANSFORMACIÓN INCREMENTAL SILVER

**Mejora implementada:** Procesamiento incremental de transformaciones

Ahora las transformaciones Bronze → Silver son **incrementales**, procesando solo los datos nuevos en lugar de reprocesar todo Bronze cada vez. Esto mejora significativamente la eficiencia del pipeline.

**Cómo funciona:**
1. Trackea el último `from_time` procesado en el JSON de estado
2. Lee de Bronze solo registros con `from_time > last_processed`
3. Aplica transformaciones solo a esos datos nuevos
4. Guarda en Silver con MERGE (evita duplicados)
5. Actualiza el estado con el nuevo `last_processed`

In [19]:

# PIPELINE DE TRANSFORMACIÓN INCREMENTAL

# PASO 1: Leer estado de transformación
state = read_state_from_json(state_file_path)
last_processed = get_last_processed_timestamp(state, 'half_hour_carbon_intensity')

if last_processed is None:
    print(f"Primera ejecución de transformación - procesando todos los datos de Bronze")
else:
    print(f"Última transformación: {last_processed}")

# PASO 2: Leer solo datos no procesados de Bronze
df_raw = read_unprocessed_data_from_bronze(
    half_hour_carbon_intensity_raw_dir,
    last_processed,
    storage_options=None
)

# PASO 3: Verificar si hay datos nuevos
if df_raw.empty:
    print(f"\nNo hay datos nuevos para procesar. Transformación completada.")
else:
    print(f"\nProcesando {len(df_raw)} registros nuevos...")
    
    # PASO 4: TRANSFORMACIONES BÁSICAS
    
    # Renombrar columnas para mayor claridad
    df_transformed = df_raw.copy()
    df_transformed.columns = df_transformed.columns.str.replace('intensity.', '', regex=False)
    df_transformed = df_transformed.rename(columns={
        'from': 'from_time',
        'to': 'to_time'
    })
    
    # Convertir fechas a datetime
    df_transformed['from_time'] = pd.to_datetime(df_transformed['from_time'])
    df_transformed['to_time'] = pd.to_datetime(df_transformed['to_time'])
    
    # PASO 5: LIMPIEZA DE DATOS
    
    # Eliminar duplicados
    df_cleaned = remove_duplicates(df_transformed, key_column='from_time', keep='last')
    
    # Eliminar valores nulos en columnas críticas
    critical_columns = ['from_time', 'to_time', 'forecast', 'actual', 'index']
    df_cleaned = remove_null_values(df_cleaned, critical_columns)
    
    # PASO 6: ENRIQUECIMIENTO DE DATOS
    
    # Agregar columnas de análisis temporal
    df_enriched = add_time_features(df_cleaned, timestamp_column='from_time')
    
    # PASO 7: COLUMNAS DE PARTICIÓN
    df_enriched['fecha'] = df_enriched['from_time'].dt.date
    
    # Resetear índice para evitar columna __index_level_0__
    df_enriched = df_enriched.reset_index(drop=True)
    
    # PASO 8: Resumen
    print(f"\nTransformaciones completadas:")
    print(f"    Registros procesados: {len(df_enriched)}")
    print(f"    Rango temporal: {df_enriched['from_time'].min()} → {df_enriched['from_time'].max()}")
    print(f"    Particiones afectadas: {df_enriched['fecha'].nunique()} días")
    print(f"    Columnas totales: {len(df_enriched.columns)}")
    
    # Mostrar muestra
    print(f"\nMuestra de datos transformados:")
    display(df_enriched.head())

Última transformación: 2025-12-01T22:30Z
Lectura incremental de Bronze
    Último procesado: 2025-12-01T22:30Z
    Filtrando particiones desde: 2025-12-01
    Registros nuevos encontrados: 217
    Rango: 2025-12-01 23:00:00+00:00 → 2025-12-06 11:00:00+00:00

Procesando 217 registros nuevos...
Detección de duplicados:
    Total registros iniciales: 217
    Duplicados encontrados: 0
    Registros eliminados: 0
    Registros finales: 217
Detección de valores nulos:
    Total registros iniciales: 217
No se encontraron valores nulos en columnas críticas
    Registros eliminados: 0
    Registros finales: 217
Columnas de tiempo agregadas: is_weekend, period_of_day
    Distribución weekend:
    Entre semana: 194 (89.4%)
    Fin de semana: 23 (10.6%)
   Distribución periodo del día:
    Noche: 102 (47.0%)
    Mañana: 59 (27.2%)
    Tarde: 56 (25.8%)

Transformaciones completadas:
    Registros procesados: 217
    Rango temporal: 2025-12-01 23:00:00+00:00 → 2025-12-06 11:00:00+00:00
    Particio

,from_time,to_time,forecast,actual,index,is_weekend,period_of_day,fecha
0,2025-12-01 23:00:00+00:00,2025-12-01 23:30:00+00:00,54,54,low,False,night,2025-12-01
1,2025-12-01 23:30:00+00:00,2025-12-02 00:00:00+00:00,54,52,low,False,night,2025-12-01
2,2025-12-02 00:00:00+00:00,2025-12-02 00:30:00+00:00,54,53,low,False,night,2025-12-02
3,2025-12-02 00:30:00+00:00,2025-12-02 01:00:00+00:00,50,51,low,False,night,2025-12-02
4,2025-12-02 01:00:00+00:00,2025-12-02 01:30:00+00:00,54,51,low,False,night,2025-12-02


In [20]:

# GUARDAR EN SILVER Y ACTUALIZAR ESTADO

# Solo guardar si hay datos nuevos procesados
if not df_raw.empty:
    # Definir ruta para capa Silver
    silver_dir = f"datalake/silver/carbon_intensity_api"
    half_hour_carbon_intensity_silver_dir = f"{silver_dir}/half_hour_carbon_intensity"
    
    print(f"\nGuardando en Silver:")
    print(f"    Ruta: {half_hour_carbon_intensity_silver_dir}")
    print(f"    Registros a guardar: {len(df_enriched)}")
    print(f"    Particionado: solo por FECHA (48 registros/partición)")
    
    # Guardar datos transformados en capa Silver con MERGE incremental
    save_new_data_as_delta(
        df_enriched, 
        half_hour_carbon_intensity_silver_dir,
        predicate="target.from_time = source.from_time",
        storage_options=None,
        partition_cols=["fecha"]
    )
    
    print(f"Datos guardados exitosamente en Silver con MERGE incremental")
    
    # Verificar que se guardó correctamente
    dt_intensity_silver = DeltaTable(half_hour_carbon_intensity_silver_dir)
    total_registros_silver = len(dt_intensity_silver.to_pandas())
    print(f"    Total registros en Silver: {total_registros_silver}")
    



    # ACTUALIZAR ESTADO DE TRANSFORMACIÓN
    
    # Obtener el último timestamp procesado
    max_timestamp = df_enriched['from_time'].max()
    
    # Actualizar estado
    update_last_processed_timestamp(
        state,
        state_file_path,
        'half_hour_carbon_intensity',
        max_timestamp
    )
    
    print(f"\nTransformación incremental completada exitosamente!")
    print(f"    Registros nuevos procesados: {len(df_enriched)}")
    print(f"    Último timestamp procesado: {max_timestamp}")
    
    # Mostrar muestra de datos en Silver
    print(f"\nMuestra de datos en Silver:")
    display(dt_intensity_silver.to_pandas().tail(5))
else:
    print(f"\nNo se requiere guardar - no hay datos nuevos")


Guardando en Silver:
    Ruta: datalake/silver/carbon_intensity_api/half_hour_carbon_intensity
    Registros a guardar: 217
    Particionado: solo por FECHA (48 registros/partición)
Datos guardados exitosamente en Silver con MERGE incremental
    Total registros en Silver: 1290
Estado de transformación actualizado: last_processed = 2025-12-06T11:00Z

Transformación incremental completada exitosamente!
    Registros nuevos procesados: 217
    Último timestamp procesado: 2025-12-06 11:00:00+00:00

Muestra de datos en Silver:


,from_time,to_time,forecast,actual,index,is_weekend,period_of_day,fecha
1285,2025-11-20 21:30:00+00:00,2025-11-20 22:00:00+00:00,218,215,high,False,night,2025-11-20
1286,2025-11-20 22:00:00+00:00,2025-11-20 22:30:00+00:00,207,216,high,False,night,2025-11-20
1287,2025-11-20 22:30:00+00:00,2025-11-20 23:00:00+00:00,209,213,high,False,night,2025-11-20
1288,2025-11-20 23:00:00+00:00,2025-11-20 23:30:00+00:00,210,211,high,False,night,2025-11-20
1289,2025-11-20 23:30:00+00:00,2025-11-21 00:00:00+00:00,215,209,high,False,night,2025-11-20


### Estrategia de Particionamiento

**Decisión:** Particionar solo por `fecha`

Me pareció que era la forma de practicar la partición, pero la verdad no sé si es la más óptima. Supongo que esto esta muy relacionado con las trasnformaciones y analisis a posterior

In [21]:
# Agregar constraints sofisticados en la capa Silver
# Ahora que los datos están transformados, podemos aplicar validaciones más estrictas
try:
    # Constraint 1: Verificar que las fechas no sean nulas
    dt_intensity_silver.alter.add_constraint(
        {"fechas_no_nulas": "from_time IS NOT NULL AND to_time IS NOT NULL"}
    )
    print("Constraint 'fechas_no_nulas' agregado")
    
    # Constraint 2: Verificar que forecast y actual sean no negativos
    dt_intensity_silver.alter.add_constraint(
        {"intensidad_no_negativa": "forecast >= 0 AND actual >= 0"}
    )
    print("Constraint 'intensidad_no_negativa' agregado")
    
    # Constraint 3: Verificar que index solo tenga valores válidos
    dt_intensity_silver.alter.add_constraint(
        {"index_valido": "index IN ('very low', 'low', 'moderate', 'high', 'very high')"}
    )
    print("Constraint 'index_valido' agregado")
    
    # Constraint 4: Verificar que la columna de partición no sea nula
    dt_intensity_silver.alter.add_constraint(
        {"particion_no_nula": "fecha IS NOT NULL"}
    )
    print("Constraint 'particion_no_nula' agregado")
    
    # Constraint 5: Verificar que la hora esté en el rango válido (0-23)
    # Nota: hora ya no es columna de partición, pero sigue siendo útil para análisis
    dt_intensity_silver.alter.add_constraint(
        {"hora_valida": "hora >= 0 AND hora <= 23"}
    )
    print("Constraint 'hora_valida' agregado")
    
    print("\nTodos los constraints agregados exitosamente a Silver")
    
except Exception as e:
    print(f"Nota al agregar constraints: {e}")

# Verificar constraints
print(f"\nTabla Silver:")
print(f"    Registros: {dt_intensity_silver.to_pandas().shape[0]}")
print(f"    Partición: solo por fecha (~48 registros/partición)")
print(f"    Columnas: {len(dt_intensity_silver.to_pandas().columns)}")

Nota al agregar constraints: Generic DeltaTable error: Constraint with name: fechas_no_nulas already exists

Tabla Silver:
    Registros: 1290
    Partición: solo por fecha (~48 registros/partición)
    Columnas: 8


## Funciones de Optimización (Compact & Vacuum)

Estas funciones ayudan a optimizar el almacenamiento de las tablas Delta Lake, reduciendo el número de archivos pequeños (small file problem) y eliminando archivos antiguos que ya no son necesarios.

In [22]:
def optimize_delta_table(table_path, storage_options=None):
    """
    Ejecuta OPTIMIZE con COMPACT en una tabla Delta Lake
    
    Combina archivos pequeños en archivos más grandes para mejorar el rendimiento
    de lectura y reducir el overhead de gestión de archivos.
    
    Parámetros:
        table_path (str): Ruta de la tabla Delta
        storage_options (dict): Opciones de almacenamiento (None para local)
    
    Retorna:
        dict: Métricas de la optimización
    """
    print(f"Optimizando tabla: {table_path}")
    
    try:
        dt = DeltaTable(table_path, storage_options=storage_options)
        
        # Contar archivos ANTES
        files_before = len(dt.file_uris())
        print(f"    Archivos antes de optimizar: {files_before}")
        
        # COMPACT: Combinar archivos pequeños
        print(f"    Ejecutando COMPACT...")
        metrics = dt.optimize.compact()
        
        # Recargar tabla y contar archivos DESPUÉS
        dt = DeltaTable(table_path, storage_options=storage_options)
        files_after = len(dt.file_uris())
        
        mejora = ((files_before - files_after) / files_before * 100) if files_before > 0 else 0
        
        print(f"Archivos después de optimizar: {files_after}")
        print(f"Reducción: {files_before - files_after} archivos ({mejora:.1f}%)")
        
        return {
            'files_before': files_before,
            'files_after': files_after,
            'reduction': files_before - files_after,
            'improvement_percentage': mejora,
            'metrics': metrics
        }
    except Exception as e:
        print(f"Error al optimizar tabla: {e}")
        return None

def vacuum_delta_table(table_path, retention_hours=168, dry_run=True, storage_options=None):
    """
    Ejecuta VACUUM para eliminar archivos antiguos de una tabla Delta Lake
    
    VACUUM elimina archivos que ya no son referenciados por la tabla y que son
    más antiguos que el periodo de retención especificado.
    
    Parámetros:
        table_path (str): Ruta de la tabla Delta
        retention_hours (int): Horas de retención (default: 168 = 7 días)
        dry_run (bool): Si True, solo muestra qué se eliminaría sin borrar
        storage_options (dict): Opciones de almacenamiento (None para local)
    
    Retorna:
        list: Lista de archivos eliminados (o a eliminar si dry_run=True)
    
    ADVERTENCIA: VACUUM es una operación IRREVERSIBLE. Siempre ejecutar con
    dry_run=True primero para verificar qué archivos se eliminarían.
    """
    print(f"\nVacuum en tabla: {table_path}")
    print(f"Retención: {retention_hours} horas ({retention_hours/24:.1f} días)")
    
    try:
        dt = DeltaTable(table_path, storage_options=storage_options)
        
        if dry_run:
            print(f"    Modo: DRY RUN (solo simulación, no se elimina nada)")
            files_to_delete = dt.vacuum(retention_hours, dry_run=True)
            print(f"    Se eliminarían {len(files_to_delete)} archivos")
            
            if len(files_to_delete) > 0:
                print(f"     Para eliminar realmente, ejecutar con dry_run=False")
            else:
                print(f"    No hay archivos para eliminar")
            
            return files_to_delete
        else:
            print(f"    Modo: REAL (eliminación permanente e irreversible)")
            deleted_files = dt.vacuum(retention_hours, dry_run=False)
            print(f"    Eliminados {len(deleted_files)} archivos permanentemente")
            
            return deleted_files
    except Exception as e:
        print(f"    Error al ejecutar vacuum: {e}")
        return []

### Aplicación de Optimizaciones

Ejecutamos COMPACT para combinar archivos pequeños y VACUUM para limpiar archivos antiguos.

In [23]:
# OPTIMIZACIÓN DE TABLAS DELTA LAKE

# Lista de tablas a optimizar
tablas_a_optimizar = [
    ("Bronze - Factors", factors_raw_dir),
    ("Bronze - Carbon Intensity", half_hour_carbon_intensity_raw_dir),
    ("Silver - Carbon Intensity", half_hour_carbon_intensity_silver_dir)
]

# Métricas totales
total_files_reduced = 0
resultados = []

# 1. COMPACT: Combinar archivos pequeños
for nombre, path in tablas_a_optimizar:
    print(f"{nombre}")
    metrics = optimize_delta_table(path, storage_options=None)
    if metrics:
        total_files_reduced += metrics['reduction']
        resultados.append({
            'tabla': nombre,
            'files_before': metrics['files_before'],
            'files_after': metrics['files_after'],
            'reduction': metrics['reduction'],
            'improvement': metrics['improvement_percentage']
        })

# 2. VACUUM: Eliminar archivos antiguos

for nombre, path in tablas_a_optimizar:
    print(f"\n{nombre}:")
    files = vacuum_delta_table(
        path,
        retention_hours=168,  # 7 días
        dry_run=True
    )

# Para ejecutar vacuum realmente, descomentar el siguiente bloque:
# for nombre, path in tablas_a_optimizar:
#     print(f"\n{nombre}:")
#     vacuum_delta_table(path, retention_hours=168, dry_run=False)

# RESUMEN

if resultados:
    print(f"\nResumen de COMPACT:")
    print(f"{'Tabla':<40} {'Antes':>10} {'Después':>10} {'Reducción':>12} {'Mejora':>10}")

    for r in resultados:
        print(f"{r['tabla']:<40} {r['files_before']:>10} {r['files_after']:>10} {r['reduction']:>12} {r['improvement']:>9.1f}%")
    
    print(f"\n{'TOTAL':<40} {'':<10} {'':<10} {total_files_reduced:>12}")

Bronze - Factors
Optimizando tabla: datalake/bronze/carbon_intensity_api/factors
    Archivos antes de optimizar: 1
    Ejecutando COMPACT...
Archivos después de optimizar: 1
Reducción: 0 archivos (0.0%)
Bronze - Carbon Intensity
Optimizando tabla: datalake/bronze/carbon_intensity_api/half_hour_carbon_intensity
    Archivos antes de optimizar: 2
    Ejecutando COMPACT...
Archivos después de optimizar: 1
Reducción: 1 archivos (50.0%)
Silver - Carbon Intensity
Optimizando tabla: datalake/silver/carbon_intensity_api/half_hour_carbon_intensity
    Archivos antes de optimizar: 29
    Ejecutando COMPACT...
Archivos después de optimizar: 28
Reducción: 1 archivos (3.4%)

Bronze - Factors:

Vacuum en tabla: datalake/bronze/carbon_intensity_api/factors
Retención: 168 horas (7.0 días)
    Modo: DRY RUN (solo simulación, no se elimina nada)
    Se eliminarían 0 archivos
    No hay archivos para eliminar

Bronze - Carbon Intensity:

Vacuum en tabla: datalake/bronze/carbon_intensity_api/half_hour_ca